# Data loading

MSGPACK to GeoParquet.

In [ ]:
from ship_routing.app.routing import RoutingResult, RoutingLog
from load_tuning_results import (
    load_results_raw,
    load_result_for_key,
    load_results,
    get_journey_params_df,
    get_hyper_params_df,
    get_runtime_df,
    get_elite_df,
    get_forcing_df,
    get_diversity_df,
)

In [ ]:
from pathlib import Path

import pandas as pd
import geopandas as gpd

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
data_files = sorted(Path("../results/").glob("results_*.msgpack"))
print(len(data_files))

In [ ]:
results = load_results(data_files)
len(results)

In [ ]:
# Load individual DataFrames
df_hyper = get_hyper_params_df(results)
df_journey = get_journey_params_df(results)
df_runtime = get_runtime_df(results)
df_elite = get_elite_df(results)
df_forcing = get_forcing_df(results)
df_diversity = get_diversity_df(results)

In [ ]:
# Merge: params (1-to-1) → runtime (1-to-1) → elite (1-to-many with left join)
df_merged = (
    df_hyper.merge(df_journey, left_index=True, right_index=True, how="inner")
    .merge(df_runtime, left_index=True, right_index=True, how="inner")
    .merge(df_elite, left_index=True, right_index=True, how="left")
    .merge(df_forcing, left_index=True, right_index=True, how="left")
    .merge(df_diversity, left_index=True, right_index=True, how="left")
)

df_merged

In [ ]:
df_merged.columns

In [ ]:
gpd.GeoDataFrame(df_merged).to_parquet("../results/results_prelim.geoparquet")